# 02 - Modelado multietiqueta

Este notebook entrena o reutiliza el modelo TF-IDF + One-vs-Rest para sugerir artículos candidatos del CEDH. La salida son puntuaciones y etiquetas candidatas revisables, no decisiones judiciales.

In [1]:
from pathlib import Path
import json, sqlite3, re, random, math, warnings, sys, subprocess, importlib.util
from datetime import datetime, timezone
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
INTERIM = DATA / 'interim'
PROCESSED = DATA / 'processed'
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
METRICS = ARTIFACTS / 'metrics'
MODELS = ARTIFACTS / 'models'
INDICES = ARTIFACTS / 'indices'
REPORTS = ARTIFACTS / 'reports'
DB = INTERIM / 'metadata.db'
SCHEMA_PATH = ROOT / 'schema.sql'
for d in [RAW, INTERIM, PROCESSED, FIGURES, METRICS, MODELS, INDICES, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Instalando {package} en este kernel...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

def dump_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('ROOT =', ROOT)

ROOT = C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado


## Definición operacional

Cada caso se representa como un documento `x_i` y una fila binaria `y_i` con 10 artículos posibles. El modelo produce un score por artículo y el ajuste de umbrales se realiza con `validation`.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, hamming_loss
import joblib

@dataclass
class OVRLogReg:
    c: float = 1.0
    max_iter: int = 2000
    solver: str = 'liblinear'
    random_state: int = 42
    n_labels: int | None = None
    constant_labels: dict = field(default_factory=dict)
    models: dict = field(default_factory=dict)
    def predict_scores(self, X):
        out = np.zeros((X.shape[0], self.n_labels), dtype=float)
        for j, v in self.constant_labels.items():
            out[:, int(j)] = float(v)
        for j, m in self.models.items():
            out[:, int(j)] = m.predict_proba(X)[:, 1]
        return out

@dataclass
class OVRLinearSVM:
    c: float = 1.0
    max_iter: int = 5000
    loss: str = 'squared_hinge'
    random_state: int = 42
    n_labels: int | None = None
    constant_labels: dict = field(default_factory=dict)
    models: dict = field(default_factory=dict)
    def predict_scores(self, X):
        out = np.zeros((X.shape[0], self.n_labels), dtype=float)
        for j, v in self.constant_labels.items():
            out[:, int(j)] = float(v)
        for j, m in self.models.items():
            margin = m.decision_function(X)
            out[:, int(j)] = 1.0 / (1.0 + np.exp(-margin))
        return out

def fit_ovr_model(model_cls, X, Y, **kwargs):
    model = model_cls(**kwargs, n_labels=Y.shape[1])
    for j in range(Y.shape[1]):
        yj = Y[:, j]
        uniques = np.unique(yj)
        if len(uniques) < 2:
            model.constant_labels[j] = int(uniques[0]) if len(uniques) else 0
            continue
        if isinstance(model, OVRLogReg):
            clf = LogisticRegression(C=model.c, max_iter=model.max_iter, solver=model.solver, random_state=model.random_state)
        else:
            clf = LinearSVC(C=model.c, max_iter=model.max_iter, loss=model.loss, random_state=model.random_state)
        clf.fit(X, yj)
        model.models[j] = clf
    return model

def load_ovr_artifact(path, cls):
    raw = joblib.load(path)
    if isinstance(raw, cls):
        return raw
    if isinstance(raw, dict):
        model = cls(n_labels=raw.get('n_labels'))
        model.constant_labels = raw.get('constant_labels', {})
        model.models = raw.get('models', {})
        return model
    raise TypeError(f'Artefacto no reconocido: {path}')

def tune_thresholds(Y_val, scores_val):
    thresholds = np.full(Y_val.shape[1], 0.5, dtype=float)
    grid = np.linspace(0.10, 0.90, 17)
    for j in range(Y_val.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            f1 = f1_score(Y_val[:, j], (scores_val[:, j] >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = t, f1
        thresholds[j] = best_t
    return thresholds

def evaluate_multilabel(Y_true, Y_pred):
    return {
        'macro_f1': float(f1_score(Y_true, Y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(Y_true, Y_pred, average='micro', zero_division=0)),
        'hamming_loss': float(hamming_loss(Y_true, Y_pred)),
    }

## Carga de matrices `X` e `Y`

La matriz `Y` conserva la naturaleza multietiqueta del problema.

In [3]:
def load_classification_frames():
    conn = sqlite3.connect(DB)
    cases_df = pd.read_sql_query('SELECT case_id, split, text_full FROM cases ORDER BY case_id', conn)
    labels_df = pd.read_sql_query('SELECT case_id, article_id FROM case_labels WHERE value=1', conn)
    articles_df = pd.read_sql_query('SELECT * FROM articles ORDER BY CAST(article_id AS INTEGER)', conn)
    conn.close()
    article_ids = articles_df['article_id'].astype(str).tolist()
    article_to_idx = {a: i for i, a in enumerate(article_ids)}
    by_case = labels_df.groupby('case_id')['article_id'].apply(lambda s: [str(x) for x in s]).to_dict()
    Y = np.zeros((len(cases_df), len(article_ids)), dtype=int)
    for i, case_id in enumerate(cases_df['case_id']):
        for article_id in by_case.get(case_id, []):
            if article_id in article_to_idx:
                Y[i, article_to_idx[article_id]] = 1
    return cases_df, Y, articles_df

cases_df, Y, articles_df = load_classification_frames()
split_counts = cases_df['split'].value_counts().reindex(['train', 'validation', 'test'])
print(split_counts.to_dict())
print('Y shape =', Y.shape, '| positivos =', int(Y.sum()))
articles_df[['article_id', 'article_code']]

{'train': 9000, 'validation': 1000, 'test': 1000}
Y shape = (11000, 10) | positivos = 15991


,article_id,article_code
0,0,2
1,1,3
2,2,5
3,3,6
4,4,8
5,5,9
6,6,10
7,7,11
8,8,14
9,9,P1-1


## Ejemplo mínimo: umbral por etiqueta

Un umbral global puede ignorar etiquetas raras. Por eso se ajusta un umbral por artículo usando la partición de validación.

In [4]:
toy_scores = np.array([[0.80, 0.20], [0.45, 0.40], [0.20, 0.70]])
toy_thresholds = np.array([0.50, 0.35])
pd.DataFrame((toy_scores >= toy_thresholds).astype(int), columns=['articulo_frecuente', 'articulo_raro'])

,articulo_frecuente,articulo_raro
0,1,0
1,0,1
2,0,1


## Entrenamiento, reutilización y artefactos

Si existen artefactos entrenados, el notebook los reutiliza para preservar reproducibilidad y regenera las predicciones en SQLite. Si faltan, entrena TF-IDF con unigramas y bigramas, Logistic Regression OVR y Linear SVM OVR.

In [5]:
def artifacts_ready():
    required = [
        MODELS / 'notebook_tfidf_vectorizer.joblib',
        MODELS / 'notebook_logreg_ovr.joblib',
        MODELS / 'notebook_svm_ovr.joblib',
        MODELS / 'notebook_thresholds.json',
    ]
    if not all(p.exists() for p in required):
        return False
    try:
        joblib.load(MODELS / 'notebook_tfidf_vectorizer.joblib')
        joblib.load(MODELS / 'notebook_logreg_ovr.joblib')
        joblib.load(MODELS / 'notebook_svm_ovr.joblib')
        read_json(MODELS / 'notebook_thresholds.json')
        return True
    except Exception as exc:
        print('Artefactos existentes no reutilizables:', exc)
        return False

def train_or_load_models(cases_df, Y):
    train_mask = cases_df['split'].eq('train').to_numpy()
    val_mask = cases_df['split'].eq('validation').to_numpy()

    if artifacts_ready():
        print('Reutilizando artefactos de modelo existentes y regenerando predicciones SQLite.')
        vectorizer = joblib.load(MODELS / 'notebook_tfidf_vectorizer.joblib')
        logreg_model = load_ovr_artifact(MODELS / 'notebook_logreg_ovr.joblib', OVRLogReg)
        svm_model = load_ovr_artifact(MODELS / 'notebook_svm_ovr.joblib', OVRLinearSVM)
        thresholds = np.array(read_json(MODELS / 'notebook_thresholds.json')['thresholds'])
        X_all = vectorizer.transform(cases_df['text_full'].fillna(''))
    else:
        print('Entrenando TF-IDF + OVR desde cero.')
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True, max_features=60000)
        X_train = vectorizer.fit_transform(cases_df.loc[train_mask, 'text_full'].fillna(''))
        X_all = vectorizer.transform(cases_df['text_full'].fillna(''))
        logreg_model = fit_ovr_model(OVRLogReg, X_train, Y[train_mask], c=1.0, max_iter=2000, solver='liblinear', random_state=SEED)
        svm_model = fit_ovr_model(OVRLinearSVM, X_train, Y[train_mask], c=1.0, max_iter=5000, loss='squared_hinge', random_state=SEED)
        thresholds = tune_thresholds(Y[val_mask], logreg_model.predict_scores(X_all)[val_mask]) if val_mask.any() else np.full(Y.shape[1], 0.5)
        joblib.dump(vectorizer, MODELS / 'notebook_tfidf_vectorizer.joblib')
        joblib.dump({'kind': 'OVRLogReg', 'n_labels': logreg_model.n_labels, 'constant_labels': logreg_model.constant_labels, 'models': logreg_model.models}, MODELS / 'notebook_logreg_ovr.joblib')
        joblib.dump({'kind': 'OVRLinearSVM', 'n_labels': svm_model.n_labels, 'constant_labels': svm_model.constant_labels, 'models': svm_model.models}, MODELS / 'notebook_svm_ovr.joblib')
        dump_json({'thresholds': thresholds.astype(float).tolist()}, MODELS / 'notebook_thresholds.json')
    return vectorizer, logreg_model, svm_model, thresholds, X_all

def persist_predictions_and_metrics(cases_df, Y, logreg_model, svm_model, thresholds, X_all):
    scores_all = logreg_model.predict_scores(X_all)
    preds_all = (scores_all >= thresholds).astype(int)
    split_order = ['train', 'validation', 'test']
    metrics_rows = []
    for split in split_order:
        idx = cases_df.index[cases_df['split'].eq(split)].to_numpy()
        metrics = evaluate_multilabel(Y[idx], preds_all[idx])
        metrics_rows.append({'model': 'tfidf_logreg_threshold_tuned', 'split': split, 'n_cases': int(len(idx)), **metrics})

    # Tabla suplementaria: comparación reproducible, no usada como tabla principal del paper.
    comparison_rows = []
    train_mask = cases_df['split'].eq('train').to_numpy()
    label_prior = Y[train_mask].mean(axis=0)
    most_common = np.zeros(Y.shape[1], dtype=int)
    most_common[int(np.argmax(label_prior))] = 1
    baseline_pred = np.tile(most_common, (Y.shape[0], 1))
    logreg_05 = (scores_all >= 0.5).astype(int)
    svm_scores = svm_model.predict_scores(X_all)
    svm_05 = (svm_scores >= 0.5).astype(int)
    candidates = [
        ('baseline_most_frequent_label', baseline_pred),
        ('tfidf_logreg_0.5', logreg_05),
        ('tfidf_svm_0.5', svm_05),
        ('tfidf_logreg_threshold_tuned', preds_all),
    ]
    for model_name, pred in candidates:
        for split in ['validation', 'test']:
            idx = cases_df.index[cases_df['split'].eq(split)].to_numpy()
            comparison_rows.append({'model': model_name, 'split': split, 'n_cases': int(len(idx)), **evaluate_multilabel(Y[idx], pred[idx])})

    rows = []
    for i, case_id in enumerate(cases_df['case_id']):
        rows.append({
            'run_id': 'notebook_threshold_tuned',
            'case_id': case_id,
            'y_true_json': json.dumps(Y[i].astype(int).tolist()),
            'y_pred_json': json.dumps(preds_all[i].astype(int).tolist()),
            'scores_json': json.dumps(scores_all[i].astype(float).tolist()),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(METRICS / 'paper_classification_table.csv', index=False)
    pd.DataFrame(comparison_rows).to_csv(METRICS / 'classification_model_comparison.csv', index=False)

    conn = sqlite3.connect(DB)
    with conn:
        conn.execute("DELETE FROM predictions WHERE run_id='notebook_threshold_tuned'")
        conn.execute("DELETE FROM experiment_runs WHERE run_id='notebook_threshold_tuned'")
        conn.executemany('INSERT OR REPLACE INTO predictions VALUES (:run_id, :case_id, :y_true_json, :y_pred_json, :scores_json)', rows)
        conn.execute(
            'INSERT OR REPLACE INTO experiment_runs VALUES (?, ?, ?, ?, ?, ?, ?)',
            (
                'notebook_threshold_tuned',
                'classification_notebook',
                'tfidf_logreg_ovr',
                json.dumps({'vectorizer': 'tfidf', 'ngram_range': [1, 2], 'thresholds': 'validation_tuned'}),
                metrics_df.to_json(orient='records'),
                datetime.now(timezone.utc).isoformat(),
                None,
            ),
        )
    conn.close()
    return metrics_df

vectorizer, logreg_model, svm_model, thresholds, X_all = train_or_load_models(cases_df, Y)
classification_metrics = persist_predictions_and_metrics(cases_df, Y, logreg_model, svm_model, thresholds, X_all)
classification_metrics

Reutilizando artefactos de modelo existentes y regenerando predicciones SQLite.


,model,split,n_cases,macro_f1,micro_f1,hamming_loss
0,tfidf_logreg_threshold_tuned,train,9000,0.816324,0.876226,0.037433
1,tfidf_logreg_threshold_tuned,validation,1000,0.654411,0.758377,0.068500
2,tfidf_logreg_threshold_tuned,test,1000,0.672007,0.736769,0.075600


## Figura final de modelado

La figura resume pipeline, métricas, distribución de scores y umbrales por etiqueta.

In [6]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import json, re, sqlite3
from pathlib import Path

COLORS = {
    'navy': '#334155',
    'teal': '#8dd3c7',
    'blue': '#80b1d3',
    'coral': '#fb8072',
    'amber': '#fdb462',
    'purple': '#bebada',
    'gray': '#64748b',
    'lightgray': '#d9d9d9',
    'offwhite': '#f8fafc',
}

def paper_style():
    sns.set_palette('Set3')
    plt.rcParams.update({
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': COLORS['navy'],
        'axes.labelcolor': COLORS['navy'],
        'axes.titlecolor': COLORS['navy'],
        'axes.titlesize': 11,
        'axes.labelsize': 9,
        'xtick.color': COLORS['navy'],
        'ytick.color': COLORS['navy'],
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'font.size': 9,
        'legend.fontsize': 8,
        'savefig.facecolor': 'white',
        'savefig.bbox': 'tight',
    })

def save_paper_figure(fig, name):
    FIGURES.mkdir(parents=True, exist_ok=True)
    out = []
    for ext in ['pdf', 'png']:
        p = FIGURES / f'{name}.{ext}'
        fig.savefig(p, dpi=240 if ext == 'png' else None, bbox_inches='tight')
        out.append(p)
    plt.close(fig)
    return out

def box(ax, xy, w, h, text, face='white', edge=None, fontsize=9, weight='normal', color=None):
    patch = FancyBboxPatch(
        xy, w, h,
        boxstyle='round,pad=0.018,rounding_size=0.018',
        linewidth=1.15,
        edgecolor=edge or COLORS['navy'],
        facecolor=face,
    )
    ax.add_patch(patch)
    ax.text(
        xy[0] + w/2, xy[1] + h/2, text,
        ha='center', va='center',
        fontsize=fontsize,
        color=color or COLORS['navy'],
        fontweight=weight,
        linespacing=1.12,
    )
    return patch

def arrow(ax, start, end, color=None, rad=0.0, lw=1.5):
    patch = FancyArrowPatch(
        start, end,
        arrowstyle='-|>',
        mutation_scale=13,
        linewidth=lw,
        color=color or COLORS['gray'],
        connectionstyle=f'arc3,rad={rad}',
    )
    ax.add_patch(patch)
    return patch

def load_project_frames():
    conn = sqlite3.connect(DB)
    cases_df = pd.read_sql_query(
        'SELECT case_id, split, year, text_full, n_paragraphs, n_tokens FROM cases',
        conn,
    )
    labels_df = pd.read_sql_query(
        'SELECT case_id, article_id, value FROM case_labels WHERE value = 1',
        conn,
    )
    articles_df = pd.read_sql_query(
        'SELECT article_id, article_code, description FROM articles ORDER BY CAST(article_id AS INTEGER)',
        conn,
    )
    conn.close()
    labels_df['article_id'] = labels_df['article_id'].astype(str)
    articles_df['article_id'] = articles_df['article_id'].astype(str)
    return cases_df, labels_df, articles_df

paper_style()
cases_df, labels_df, articles_df = load_project_frames()
metrics = pd.read_csv(METRICS / 'paper_classification_table.csv')
thresholds = np.array(read_json(MODELS / 'notebook_thresholds.json')['thresholds'])
conn = sqlite3.connect(DB)
preds = pd.read_sql_query(
    "SELECT p.case_id, p.scores_json, c.split "
    "FROM predictions p "
    "JOIN cases c ON c.case_id = p.case_id "
    "WHERE p.run_id='notebook_threshold_tuned' "
    "ORDER BY p.case_id",
    conn,
)
conn.close()
preds['scores'] = preds['scores_json'].map(lambda s: np.array(json.loads(s), dtype=float))

fig = plt.figure(figsize=(13, 7))
gs = GridSpec(2, 2, figure=fig, hspace=0.36, wspace=0.28)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

ax_a.axis('off')
ax_a.set_xlim(0, 1)
ax_a.set_ylim(0, 1)
xs = [0.13, 0.39, 0.65, 0.88]
texts = ['Texto\njurídico', 'TF-IDF\n1-2 gramas', 'OVR\nLogReg', 'Scores +\numbrales']
faces = [COLORS['offwhite'], '#eef2ff', '#fefce8', '#ecfeff']
for x, text, face in zip(xs, texts, faces):
    box(ax_a, (x - 0.095, 0.43), 0.19, 0.16, text, face, fontsize=9, weight='bold')
for i in range(3):
    arrow(ax_a, (xs[i] + 0.095, 0.51), (xs[i + 1] - 0.095, 0.51), COLORS['teal'])
ax_a.set_title('A. Pipeline de modelado')

pivot = metrics.pivot_table(index='split', values=['macro_f1', 'micro_f1', 'hamming_loss'], aggfunc='first').reindex(['train','validation','test'])
x = np.arange(len(pivot.index))
ax_b.bar(x - 0.23, pivot['macro_f1'], width=0.22, label='macro-F1', color=COLORS['teal'])
ax_b.bar(x, pivot['micro_f1'], width=0.22, label='micro-F1', color=COLORS['blue'])
ax_b.bar(x + 0.23, pivot['hamming_loss'], width=0.22, label='Hamming loss', color=COLORS['coral'])
ax_b.set_xticks(x)
ax_b.set_xticklabels(pivot.index)
ax_b.set_ylim(0, 1)
ax_b.set_title('B. Métricas por split')
ax_b.legend(frameon=False, ncol=3, loc='upper right')

test_scores = np.vstack(preds[preds['split'].eq('test')]['scores'].to_numpy())
parts = ax_c.violinplot([test_scores[:, i] for i in range(test_scores.shape[1])], showmeans=True, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor(COLORS['blue'])
    pc.set_edgecolor(COLORS['blue'])
    pc.set_alpha(0.35)
parts['cmeans'].set_color(COLORS['navy'])
ax_c.set_xticks(range(1, len(articles_df) + 1))
ax_c.set_xticklabels(articles_df['article_code'], rotation=45, ha='right')
ax_c.set_ylim(0, 1)
ax_c.set_ylabel('score')
ax_c.set_title('C. Distribución de scores por artículo, test')

ax_d.bar(articles_df['article_code'], thresholds, color=COLORS['amber'])
ax_d.set_ylim(0, 1)
ax_d.set_ylabel('umbral')
ax_d.set_title('D. Umbral ajustado por etiqueta')
ax_d.tick_params(axis='x', rotation=45)
for i, value in enumerate(thresholds):
    ax_d.text(i, value + 0.025, f'{value:.2f}', ha='center', fontsize=7, color=COLORS['navy'])

fig.suptitle('Figura 4. Modelado multietiqueta y ajuste de umbrales', fontsize=14, fontweight='bold', color=COLORS['navy'])
written = save_paper_figure(fig, 'fig04_modeling_and_thresholds')
print('\n'.join(str(p) for p in written))

C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig04_modeling_and_thresholds.pdf
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig04_modeling_and_thresholds.png
